# Level 1 Baseline: SVM Acoustic Gunshot Detection
This notebook establishes a Level 1 Baseline using a Support Vector Machine (SVM) on 13 MFCC features. This allows us to prove our dataset is clean and establish a performance floor before moving to complex Deep Learning architectures.

## 1. Dependencies
Run the cell below to install any missing dependencies, then import the required libraries.

In [1]:
%pip install -q scikit-learn librosa pandas numpy matplotlib seaborn

import os
import numpy as np
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

Note: you may need to restart the kernel to use updated packages.


## 2. Feature Extraction Engine
This function loads an audio file, resamples it to 16 kHz (standard for many edge/speech applications), and extracts 13 Mel-Frequency Cepstral Coefficients (MFCCs). We take the mean across the time axis to get a 1D feature vector per audio clip.

In [2]:
def extract_mfcc_features(file_path):
    try:
        # Load audio at 16000 Hz target sample rate
        y, sr = librosa.load(file_path, sr=16000)
        if len(y) == 0:
            return None
        
        # Extract 13 MFCCs
        mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
        
        # Take the mean across the time axis
        mfccs_scaled = np.mean(mfccs.T, axis=0)
        return mfccs_scaled
        
    except Exception as e:
        print(f"Error processing corrupted file {file_path}: {e}")
        return None

## 3. Data Loading
Here we iterate through our two dummy directories for Class 1 (Gunshots) and Class 0 (Background Noise). We extract features and append them to our `X` feature array and `y` label array.

In [3]:
class_1_dir = './Data/Gunshots/'
class_0_dir = './Data/Others/'

X_list = []
y_list = []

# Load Class 1 (Gunshots)
if os.path.exists(class_1_dir):
    for f in os.listdir(class_1_dir):
        if f.endswith('.wav'):
            path = os.path.join(class_1_dir, f)
            feats = extract_mfcc_features(path)
            if feats is not None:
                X_list.append(feats)
                y_list.append(1)

# Load Class 0 (Background Noise)
if os.path.exists(class_0_dir):
    for f in os.listdir(class_0_dir):
        if f.endswith('.wav'):
            path = os.path.join(class_0_dir, f)
            feats = extract_mfcc_features(path)
            if feats is not None:
                X_list.append(feats)
                y_list.append(0)

X = np.array(X_list)
y = np.array(y_list)

print(f"Successfully loaded {len(X)} audio files.")
if len(y) > 0:
    unique, counts = np.unique(y, return_counts=True)
    print(f"Class distribution: {dict(zip(unique, counts))}")

d:\anaconda3\envs\LLM\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Successfully loaded 50 audio files.
Class distribution: {np.int64(1): np.int64(50)}


## 4. The Data Split & Scale (Critical)
We split the data 80/20, using `stratify=y` to maintain the 9-to-1 class imbalance ratio in both the training and test sets. We then strictly fit the `StandardScaler` on the training data only, and use it to transform both sets to prevent data leakage.

In [4]:
if len(X) > 5 and len(np.unique(y)) > 1:
    # Split 80/20 with stratification
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Initialize StandardScaler
    scaler = StandardScaler()
    
    # Fit and transform on training data ONLY
    X_train_scaled = scaler.fit_transform(X_train)
    
    # Only transform the test data (prevent data leakage)
    X_test_scaled = scaler.transform(X_test)
    
    print("Data split and scaled successfully!")
else:
    print("Not enough data or missing classes to perform train/test split. Ensure your dummy directories are populated.")

Not enough data or missing classes to perform train/test split. Ensure your dummy directories are populated.


## 5. The Model (Handling Class Imbalance)
Because we have a severe class imbalance (mostly background noise), we initialize our Support Vector Classifier with `class_weight='balanced'`. This penalizes the model heavily for missing the minority gunshot class.

In [5]:
if "X_train_scaled" in locals():
    # Initialize SVM with RBF kernel and balanced class weights
    svm_model = SVC(kernel='rbf', class_weight='balanced', random_state=42)
    
    # Train the SVM
    svm_model.fit(X_train_scaled, y_train)
    print("SVM Model training complete!")

## 6. Evaluation Visualization
We predict on the test set, output the precision/recall metrics via `classification_report`, and plot an annotated heatmap of the `confusion_matrix` to visually inspect False Positives vs. False Negatives.

In [7]:
if "svm_model" in locals():
    # Predict on the scaled test set
    y_pred = svm_model.predict(X_test_scaled)
    
    # Print classification report
    print("Classification Report:\n")
    print(classification_report(y_test, y_pred, target_names=['Background (0)', 'Gunshot (1)']))
    
    # Generate Confusion Matrix Heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Background', 'Gunshot'], yticklabels=['Background', 'Gunshot'])
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('SVM Confusion Matrix', fontsize=14)
    plt.show()